# Feature Engineering

This notebook creates features for two machine-learning tasks:

1. Solar-power forecasting.
2. Solar-inverter anomaly detection.

The processed datasets are used as inputs, and final feature datasets are saved in the `data/processed` folder.

## 1. Forecasting features

Load the cleaned DKASC forecast data and create:

- Hour, month, and day-of-week features.

- Solar zenith and elevation features.**Output:** `forecast_features_final.csv`

- 1-hour, 24-hour, and rolling production features.

In [6]:
import pandas as pd
import pvlib
import os

PROCESSED_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast'
clean_path = os.path.join(PROCESSED_DIR, 'forecast_dkasc_clean.csv')
output_path = os.path.join(PROCESSED_DIR, 'forecast_features_final.csv')

print("Loading cleaned forecast data...")
df = pd.read_csv(clean_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print("Adding time features...")
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['day_of_week'] = df['timestamp'].dt.dayofweek

print("Adding solar position features...")
site = pvlib.location.Location(-23.76, 133.87, tz='Australia/Darwin')
solar_position = site.get_solarposition(df['timestamp'])
df['solar_zenith'] = solar_position['zenith'].values
df['solar_elevation'] = solar_position['elevation'].values

print("Adding lag and rolling features...")
df['power_lag_1h'] = df['actual_power_kw'].shift(12).fillna(0)
df['power_lag_24h'] = df['actual_power_kw'].shift(288).fillna(0)
df['power_roll_mean_1h'] = df['actual_power_kw'].rolling(window=12).mean().fillna(0)

df.to_csv(output_path, index=False)
print(f"\nFeature engineering completed. Number of rows: {len(df)}")
print(f"Saved file: {output_path}")
print("\nSample of the final dataset:")
print(df[['timestamp', 'actual_power_kw', 'ghi', 'solar_zenith', 'power_lag_1h']].head())

Loading cleaned forecast data...
Adding time features...
Adding solar position features...
Adding lag and rolling features...

Feature engineering completed. Number of rows: 98146
Saved file: F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast\forecast_features_final.csv

Sample of the final dataset:
            timestamp  actual_power_kw       ghi  solar_zenith  power_lag_1h
0 2025-01-01 00:00:00              0.0  2.122361     42.939074           0.0
1 2025-01-01 00:05:00              0.0  2.309545     41.808503           0.0
2 2025-01-01 00:10:00              0.0  2.226635     40.676999           0.0
3 2025-01-01 00:15:00              0.0  2.177083     39.544599           0.0
4 2025-01-01 00:20:00              0.0  2.419703     38.411337           0.0


## 2. Kaggle anomaly features

Load the labeled Kaggle dataset and create:

- Hour and day-of-week features.

- Relative inverter power ratio.**Output:** `anomaly_features_final.csv`

- Actual-minus-expected power residual.
- Previous 1-step and 4-step inverter power values.

In [ ]:
import pandas as pd
import os

PROCESSED_DIR = 'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed'
input_path = os.path.join(PROCESSED_DIR, 'kaggle_anomaly_data.csv')
output_path = os.path.join(PROCESSED_DIR, 'anomaly_features_final.csv')

print("Loading anomaly data...")
df = pd.read_csv(input_path)
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])

df = df.sort_values(['SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

print("Adding time features...")
df['hour'] = df['DATE_TIME'].dt.hour
df['day_of_week'] = df['DATE_TIME'].dt.dayofweek

print("Calculating inverter production ratio...")
plant_avg_power = df.groupby('DATE_TIME')['AC_POWER'].transform('mean')
df['inverter_ratio'] = df['AC_POWER'] / (plant_avg_power + 1e-5)

print("Calculating actual versus expected power residual...")
expected_power = df.groupby('IRRADIATION')['AC_POWER'].transform('median')
df['residual'] = df['AC_POWER'] - expected_power

print("Adding inverter lag features...")
df['power_lag_1'] = df.groupby('SOURCE_KEY')['AC_POWER'].shift(1)
df['power_lag_4'] = df.groupby('SOURCE_KEY')['AC_POWER'].shift(4)
df[['power_lag_1', 'power_lag_4']] = df[['power_lag_1', 'power_lag_4']].fillna(0)

df = df.dropna(subset=['inverter_ratio', 'residual'])

cols_to_drop = ['SOURCE_KEY_weather', 'DAILY_YIELD', 'TOTAL_YIELD', 'PLANT_ID']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"\nFeature engineering completed. Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
print("\nSample of the new data:")
print(df[['DATE_TIME', 'SOURCE_KEY', 'AC_POWER', 'inverter_ratio', 'residual', 'is_anomaly']].head(10))

df.to_csv(output_path, index=False)
print(f"\nSaved the model-ready anomaly data to: {output_path}")